In [2]:
"""
Gurgaon (Gurugram) Sectors - Latitude & Longitude Scraper
Uses Nominatim (OpenStreetMap) via geopy — no API key required.
Exports results to a CSV and returns a pandas DataFrame.
"""

import time
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# ── Configuration ─────────────────────────────────────────────────────────────

# Gurgaon has sectors 1–115 (not all are named/populated; some are under dev.)
SECTORS = list(range(1, 116))

GEOLOCATOR = Nominatim(user_agent="gurgaon_sector_geocoder_v1", timeout=10)

# Nominatim's free tier: max 1 request/second (required by their ToS)
REQUEST_DELAY = 1.1  # seconds between requests

OUTPUT_CSV = "gurgaon_sectors_latlong.csv"


# ── Geocoding function ─────────────────────────────────────────────────────────

def geocode_sector(sector_num: int, retries: int = 3) -> dict:
    """
    Try multiple query formats for a sector and return the best hit.
    Returns a dict with sector, query, latitude, longitude, display_name.
    """
    queries = [
        f"Sector {sector_num}, Gurugram, Haryana, India",
        f"Sector {sector_num}, Gurgaon, Haryana, India",
        f"Sector-{sector_num} Gurugram India",
    ]

    for query in queries:
        for attempt in range(retries):
            try:
                location = GEOLOCATOR.geocode(query)
                if location:
                    return {
                        "sector":       sector_num,
                        "query_used":   query,
                        "latitude":     round(location.latitude, 6),
                        "longitude":    round(location.longitude, 6),
                        "display_name": location.address,
                        "status":       "found",
                    }
                break  # query returned None → try next query format
            except GeocoderTimedOut:
                print(f"  Timeout on Sector {sector_num} (attempt {attempt+1}), retrying…")
                time.sleep(2)
            except GeocoderServiceError as e:
                print(f"  Service error on Sector {sector_num}: {e}")
                time.sleep(3)
        time.sleep(REQUEST_DELAY)

    # Nothing found after all queries
    return {
        "sector":       sector_num,
        "query_used":   None,
        "latitude":     None,
        "longitude":    None,
        "display_name": None,
        "status":       "not_found",
    }


# ── Main ───────────────────────────────────────────────────────────────────────

def main():
    print(f"Geocoding {len(SECTORS)} Gurgaon sectors (this takes ~{len(SECTORS)*REQUEST_DELAY/60:.1f} min)…\n")

    results = []
    for i, sector in enumerate(SECTORS, 1):
        print(f"[{i:3}/{len(SECTORS)}] Sector {sector:>3}…", end=" ", flush=True)
        row = geocode_sector(sector)
        results.append(row)
        print(f"{'✓' if row['status']=='found' else '✗'}  {row['latitude']}, {row['longitude']}")
        time.sleep(REQUEST_DELAY)

    df = pd.DataFrame(results)

    # Summary
    found    = (df["status"] == "found").sum()
    notfound = (df["status"] == "not_found").sum()
    print(f"\n{'─'*55}")
    print(f"  Found   : {found} sectors")
    print(f"  Not found: {notfound} sectors")
    print(f"{'─'*55}\n")

    # Export
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved → {OUTPUT_CSV}\n")
    print(df[df["status"] == "found"][["sector", "latitude", "longitude"]].to_string(index=False))

    return df


if __name__ == "__main__":
    df = main()

Geocoding 115 Gurgaon sectors (this takes ~2.1 min)…

[  1/115] Sector   1… ✓  28.360102, 76.948375
[  2/115] Sector   2… ✓  28.50905, 77.034283
[  3/115] Sector   3… ✓  28.497391, 77.020526
[  4/115] Sector   4… ✓  28.475006, 77.010353
[  5/115] Sector   5… ✓  28.367354, 76.926724
[  6/115] Sector   6… ✓  28.475326, 77.025828
[  7/115] Sector   7… ✓  28.466257, 77.014265
[  8/115] Sector   8… ✓  28.459897, 77.019804
[  9/115] Sector   9… ✓  28.384099, 76.886134
[ 10/115] Sector  10… ✓  28.444581, 77.006023
[ 11/115] Sector  11… ✓  28.378143, 76.875377
[ 12/115] Sector  12… ✓  28.392411, 76.898514
[ 13/115] Sector  13… ✓  28.4746, 77.036719
[ 14/115] Sector  14… ✓  28.473784, 77.047183
[ 15/115] Sector  15… ✓  28.38432, 76.912233
[ 16/115] Sector  16… ✓  28.467613, 77.05108
[ 17/115] Sector  17… ✓  28.475804, 77.060777
[ 18/115] Sector  18… ✓  28.491345, 77.071026
[ 19/115] Sector  19… ✓  28.503091, 77.081748
[ 20/115] Sector  20… ✓  28.508496, 77.087939
[ 21/115] Sector  21… ✓  28.514

In [11]:
df.head()

,sector,latitude,longitude
0,1,28.360102,76.948375
1,2,28.509050,77.034283
2,3,28.497391,77.020526
3,4,28.475006,77.010353
4,5,28.367354,76.926724


In [12]:
df.to_csv("geographical_data.csv")

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 115 entries, 0 to 114
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sector        115 non-null    int64  
 1   query_used    115 non-null    str    
 2   latitude      115 non-null    float64
 3   longitude     115 non-null    float64
 4   display_name  115 non-null    str    
 5   status        115 non-null    str    
dtypes: float64(2), int64(1), str(3)
memory usage: 15.3 KB


In [10]:
df.drop(columns=["query_used","display_name","status"],inplace = True)

KeyError: "['query_used', 'display_name', 'status'] not found in axis"

In [8]:
df.to_csv("geographical-data.csv")